# $B^\pm\to K^\pm\pi^+\pi^-$ direct-CP toy fit

Generate charge-conjugate signal toys and fit them simultaneously with the joint charge-Dalitz likelihood. The particle ordering is $(K^\pm,\pi^\pm,\pi^\mp)$, and all Dalitz plots/projections use $s_{13}=m^2(K^\pm\pi^\mp)$ and $s_{23}=m^2(\pi^+\pi^-)$.


In [ ]:
import numpy as np
import jax
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    BaBarFlatte, CPRealImag, DecayChannel, DecayModel, LASS, Minimizer,
    NonResonant, Parameter, Resonance, enable_x64, weighted_resample,
)
from dalitzplotfitter.likelihood import CPJointNLL

enable_x64()


## 1. Shared CP coefficients and charge-conjugate models


In [ ]:
truth_spec = {
    "Kstar892": (1.00, 0.00, +0.04, -0.03),
    "KpiS":     (1.40, -0.60, -0.10, +0.08),
    "rho770":   (0.65, 0.10, +0.06, +0.04),
    "f0_980":   (-0.20, 1.00, -0.05, +0.07),
    "NR":       (-0.50, 0.10, 0.00, 0.00),
}
truth = {}
shared = {}

for name, (x, y, dx, dy) in truth_spec.items():
    xpar = Parameter.coefficient(f"{name}.x", x, owner=name, fixed=(name == "Kstar892"), step=0.01)
    ypar = Parameter.coefficient(f"{name}.y", y, owner=name, fixed=(name == "Kstar892"), step=0.01)
    dxpar = Parameter.coefficient(f"{name}.dx", dx, owner=name, fixed=(name == "NR"), step=0.01)
    dypar = Parameter.coefficient(f"{name}.dy", dy, owner=name, fixed=(name == "NR"), step=0.01)
    shared[name] = CPRealImag(xpar, ypar, dxpar, dypar)
    for p in (xpar, ypar, dxpar, dypar):
        truth[p.name] = p.value

def components_for_charge(charge):
    c = {name: coeff.for_charge(charge) for name, coeff in shared.items()}
    return [
        Resonance("Kstar892", (0,2), c["Kstar892"], mass=0.8958, width=0.0474, spin=1, resonance_radius=4.0, parent_radius=4.0),
        Resonance("KpiS", (0,2), c["KpiS"], lineshape=LASS(2.07, 3.32, 1.8), mass=1.425, width=0.270, spin=0, resonance_radius=4.0, parent_radius=4.0),
        Resonance("rho770", (1,2), c["rho770"], mass=0.7753, width=0.1491, spin=1, resonance_radius=4.0, parent_radius=4.0),
        Resonance("f0_980", (1,2), c["f0_980"], lineshape=BaBarFlatte(), mass=0.965, width=0.0, spin=0, resonance_radius=4.0, parent_radius=4.0),
        NonResonant(c["NR"]),
    ]

plus_model = DecayModel(
    DecayChannel("B+", ("K+", "pi+", "pi-")),
    components_for_charge(+1),
    normalization_method="square-dalitz",
    normalization_resolution=350,
    normalization_pair=(0, 2),
)
minus_model = DecayModel(
    DecayChannel("B-", ("K-", "pi-", "pi+")),
    components_for_charge(-1),
    normalization_method="square-dalitz",
    normalization_resolution=350,
    normalization_pair=(0, 2),
)
plus_norm = plus_model.normalization_sample
minus_norm = minus_model.normalization_sample
fit_parameters = tuple(p for p in plus_model.parameters if not p.fixed)
print("free parameters:", len(fit_parameters))


## 2. Generate a joint charge-Dalitz pseudoexperiment


In [ ]:
N_POOL = 200_000
N_DATA = 40_000

plus_pool = plus_model.generate_phase_space(N_POOL, seed=5001)
minus_pool = minus_model.generate_phase_space(N_POOL, seed=5002)
plus_pool_cache = plus_model.prepare_cache(plus_pool, plus_norm)
minus_pool_cache = minus_model.prepare_cache(minus_pool, minus_norm)

i_plus = float(plus_pool_cache.normalization(truth))
i_minus = float(minus_pool_cache.normalization(truth))
p_plus = i_plus / (i_plus + i_minus)

rng = np.random.default_rng(5003)
n_plus = rng.binomial(N_DATA, p_plus)
n_minus = N_DATA - n_plus

plus_data = weighted_resample(
    jax.random.key(5004), plus_pool,
    plus_pool.weights * plus_pool_cache.intensity(truth),
    n_plus, replace=True,
)
minus_data = weighted_resample(
    jax.random.key(5005), minus_pool,
    minus_pool.weights * minus_pool_cache.intensity(truth),
    n_minus, replace=True,
)

print("generated B+/B-:", n_plus, n_minus)
print("truth charge probabilities:", p_plus, 1.0-p_plus)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
for ax, sample, title in ((axes[0], plus_data, "B+"), (axes[1], minus_data, "B-")):
    h = ax.hist2d(np.asarray(sample.s13), np.asarray(sample.s23), bins=80)
    fig.colorbar(h[3], ax=ax, label="events")
    ax.set(xlabel=r"$s_{13}$ [GeV$^2$]", ylabel=r"$s_{23}$ [GeV$^2$]", title=title)
plt.show()


## 3. Joint CP fit


In [ ]:
plus_cache = plus_model.prepare_cache(plus_data, plus_norm)
minus_cache = minus_model.prepare_cache(minus_data, minus_norm)
nll = CPJointNLL(plus_cache, minus_cache)

rng = np.random.default_rng(5006)
start = {p.name: truth[p.name] + rng.normal(0.0, 0.08) for p in fit_parameters}
result = Minimizer(nll, fit_parameters, verbose=1).fit(
    start_values=start, simplex=True, ncall=50_000
)
fit_values = {p.name: float(result.values[p.name]) for p in fit_parameters}
pfit_plus, pfit_minus = nll.charge_probabilities(fit_values)

print("valid:", result.valid, "NLL:", result.fval, "EDM:", result.fmin.edm)
print("charge probabilities truth:", p_plus, 1.0-p_plus)
print("charge probabilities fit:  ", float(pfit_plus), float(pfit_minus))
print(f"{'parameter':18s} {'generated':>11s} {'fitted':>11s} {'error':>11s} {'pull':>9s}")
for p in fit_parameters:
    fitted = float(result.values[p.name])
    error = float(result.errors[p.name])
    pull = (fitted-truth[p.name])/error
    print(f"{p.name:18s} {truth[p.name]:11.5f} {fitted:11.5f} {error:11.5f} {pull:9.3f}")


## 4. Charge-separated $s_{13}$ and $s_{23}$ projections


In [ ]:
def projection(pool, cache, values, variable, bins):
    weights = np.asarray(pool.weights * cache.intensity(values))
    hist, _ = np.histogram(np.asarray(getattr(pool, variable)), bins=bins, weights=weights)
    return hist

fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
for row, (sample, pool, cache, title) in enumerate((
    (plus_data, plus_pool, plus_pool_cache, "B+"),
    (minus_data, minus_pool, minus_pool_cache, "B-"),
)):
    for col, (variable, label) in enumerate((("s13", r"$s_{13}$ [GeV$^2$]"), ("s23", r"$s_{23}$ [GeV$^2$]"))):
        ax = axes[row, col]
        observed = np.asarray(getattr(sample, variable))
        bins = np.linspace(observed.min(), observed.max(), 70)
        centers = 0.5*(bins[:-1] + bins[1:])
        data_hist, _ = np.histogram(observed, bins=bins)
        truth_hist = projection(pool, cache, truth, variable, bins)
        fit_hist = projection(pool, cache, fit_values, variable, bins)
        truth_hist *= data_hist.sum()/truth_hist.sum()
        fit_hist *= data_hist.sum()/fit_hist.sum()
        ax.errorbar(centers, data_hist, yerr=np.sqrt(np.maximum(data_hist, 1)), fmt=".", label="toy data")
        ax.step(centers, truth_hist, where="mid", linestyle="--", label="truth")
        ax.step(centers, fit_hist, where="mid", label="fit")
        ax.set(xlabel=label, ylabel="events / bin", title=title)
        ax.legend()
plt.show()


## Interpretation

The likelihood is normalized over the combined charge-Dalitz space, so the fit retains both local CP-sensitive interference information and the integrated $B^+/B^-$ rate asymmetry.
